# Task 3 — Two-stage Hurdle GBM + ZILN (Monthly Spending, NMAE)
**ไอเดีย:** ZILN loss = BCE(จะจ่ายไหม) + Lognormal NLL(จ่ายเท่าไร) → แยกเป็นโมเดล GBM ได้ตรง ๆ
- Stage 1: XGB classifier → p = P(y>0)
- Stage 2a: XGB regressor บน log1p(y) (เฉพาะคนจ่าย) → μ
- Stage 2b: XGB regressor บน residual² → σ ต่อแถว
- Stage 2c: XGB multi-quantile (log1p) → การกระจายแบบไม่สมมติ lognormal

**NMAE = MAE → ค่าที่ดีที่สุดคือ median ของ mixture:**  p ≤ 0.5 → 0 · p > 0.5 → quantile ที่ q = 1 − 1/(2p) ของคนจ่าย

เทียบด้วย 5-fold OOF: A) MAE ตรง · B) hurdle-median · C) ZILN-quantile · D) multi-quantile · E) blend
XGB: 4000 trees, lr 0.003, early stop 100 (บน inner holdout 10% ของ train fold → OOF ไม่รั่ว)
**ไม่ submit** — เขียนไฟล์ลง `/kaggle/working/` เท่านั้น

In [ ]:
import os, time, json, warnings, numpy as np, pandas as pd, xgboost as xgb
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold, train_test_split
warnings.filterwarnings("ignore")
SEED, NF = 42, 5
N_TREES, LR, ES = 4000, 0.003, 100
DEVICE = "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
print("xgboost", xgb.__version__, "| device", DEVICE)

def walk_find(name, must=None):
    for d, _, fs in os.walk("/kaggle/input", followlinks=True):
        if name in fs and (must is None or must in d): return os.path.join(d, name)
TRAIN, TEST = walk_find("train.csv", "03_regression"), walk_find("test.csv", "03_regression")
SAMPLE = walk_find("sample_submission.csv")
print(TRAIN, TEST, SAMPLE, sep="\n")
assert TRAIN and TEST, "ไม่เจอข้อมูล — กด Add Input → competition data"
tr, te = pd.read_csv(TRAIN), pd.read_csv(TEST)
TARGET = "spending_30d"; y = tr[TARGET].values.astype(float)
ids = pd.read_csv(SAMPLE)["id"].values if SAMPLE else np.array([f"ANS{i+1:05d}" for i in range(len(te))])
assert len(ids) == len(te)
print(tr.shape, te.shape, "| y=0:", round((y == 0).mean(), 3), "| mean", round(y.mean(), 1), "| max", y.max())
def nmae(yt, yp): return 1 / (1 + np.abs(yt - yp).sum() / yt.sum())

## 1. Features

In [ ]:
SPEND = ["historical_spending", "prev_month_spending", "avg_transaction_value"]
def add_feats(d):
    d = d.copy(); e = 1.0
    d["hist_per_day"]      = d["historical_spending"] / (d["account_age_days"] + e)
    d["tx_per_day"]        = d["total_transactions"] / (d["account_age_days"] + e)
    d["exp_monthly_spend"] = d["avg_transaction_value"] * d["purchase_frequency"]
    d["prev_vs_hist"]      = d["prev_month_spending"] / (d["historical_spending"] + e)
    d["hist_per_tx"]       = d["historical_spending"] / (d["total_transactions"] + e)
    d["prev_x_freq"]       = d["prev_month_spending"] * d["purchase_frequency"]
    d["recency_ratio"]     = d["days_since_last_purchase"] / (d["days_since_last_login"] + e)
    d["play_per_session"]  = d["total_playtime_hours"] / (d["sessions_per_week"] + e)
    d["disc_share"]        = d["purchases_on_discount"] / (d["total_transactions"] + e)
    for c in SPEND: d[f"log_{c}"] = np.log1p(d[c].clip(lower=0))
    d["n_missing"] = d.isna().sum(1)
    return d
X, Xt = add_feats(tr.drop(columns=[TARGET])), add_feats(te)
FEATS = [c for c in Xt.columns if c != "player_id"]
X, Xt = X[FEATS].copy(), Xt[FEATS].copy()
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(X[c])]
for c in CATS:
    cats = sorted(set(X[c].dropna().astype(str)) | set(Xt[c].dropna().astype(str)))
    X[c] = pd.Categorical(X[c].astype("object").where(X[c].notna(), None), categories=cats)
    Xt[c] = pd.Categorical(Xt[c].astype("object").where(Xt[c].notna(), None), categories=cats)
print(len(FEATS), "features | categorical:", CATS)

## 2. Models

In [ ]:
BASE = dict(n_estimators=N_TREES, learning_rate=LR, early_stopping_rounds=ES, max_depth=7, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0, tree_method="hist", device=DEVICE,
            enable_categorical=True, max_cat_to_onehot=8, random_state=SEED)
ALPHAS = np.array([0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6])

def fit_es(model, Xa, ya, strat=None):
    """early stop บน inner holdout 10% ของ train fold"""
    i_tr, i_es = train_test_split(np.arange(len(ya)), test_size=0.1, random_state=SEED, stratify=strat)
    model.fit(Xa.iloc[i_tr], ya[i_tr], eval_set=[(Xa.iloc[i_es], ya[i_es])], verbose=False)
    return model

pos_all = y > 0
strat = np.where(pos_all, 1 + pd.qcut(pd.Series(y).where(pos_all).rank(method="first"), 4, labels=False).fillna(-1).values, 0).astype(int)
folds = list(StratifiedKFold(NF, shuffle=True, random_state=SEED).split(X, strat))
n, nt = len(y), len(Xt)
O = {k: np.zeros(n) for k in ["mae", "p", "mu", "sig"]}; O["q"] = np.zeros((n, len(ALPHAS)))
T = {k: np.zeros(nt) for k in ["mae", "p", "mu", "sig"]}; T["q"] = np.zeros((nt, len(ALPHAS)))
best_it = {}

In [ ]:
t_all = time.time()
for k, (a, b) in enumerate(folds):
    t0 = time.time(); Xa, Xb, ya = X.iloc[a], X.iloc[b], y[a]; pa = ya > 0
    # A) MAE ตรง ๆ ทุกแถว
    m = fit_es(xgb.XGBRegressor(objective="reg:absoluteerror", **BASE), Xa, ya, strat[a])
    O["mae"][b] = m.predict(Xb); T["mae"] += m.predict(Xt) / NF; best_it.setdefault("mae", []).append(m.best_iteration)
    # Stage 1) จะจ่ายไหม
    c = fit_es(xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss", **BASE), Xa, pa.astype(int), pa.astype(int))
    O["p"][b] = c.predict_proba(Xb)[:, 1]; T["p"] += c.predict_proba(Xt)[:, 1] / NF; best_it.setdefault("clf", []).append(c.best_iteration)
    # Stage 2a) μ บน log1p(y) เฉพาะคนจ่าย
    Xp, lp = Xa[pa], np.log1p(ya[pa])
    r = fit_es(xgb.XGBRegressor(objective="reg:squarederror", **BASE), Xp, lp)
    O["mu"][b] = r.predict(Xb); T["mu"] += r.predict(Xt) / NF; best_it.setdefault("mu", []).append(r.best_iteration)
    # Stage 2b) σ ต่อแถว จาก residual² (in-sample → ขยาย scale ทีหลังด้วย OOF)
    res2 = (lp - r.predict(Xp)) ** 2
    s = fit_es(xgb.XGBRegressor(objective="reg:squarederror", **{**BASE, "max_depth": 5}), Xp, res2)
    sg = lambda Z: np.sqrt(np.clip(s.predict(Z), 1e-4, None))
    O["sig"][b] = sg(Xb); T["sig"] += sg(Xt) / NF; best_it.setdefault("sig", []).append(s.best_iteration)
    # Stage 2c) multi-quantile บน log1p(y)
    qm = fit_es(xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=ALPHAS, **BASE), Xp, lp)
    O["q"][b] = np.sort(qm.predict(Xb), 1); T["q"] += np.sort(qm.predict(Xt), 1) / NF; best_it.setdefault("q", []).append(qm.best_iteration)
    print(f"fold {k}: {time.time()-t0:.0f}s | best_iter " + ", ".join(f"{kk}={v[-1]}" for kk, v in best_it.items())
          + f" | fold NMAE(mae)={nmae(y[b], np.clip(O['mae'][b],0,None)):.4f}")
print(f"total {time.time()-t_all:.0f}s")
if any(max(v) >= N_TREES - ES - 1 for v in best_it.values()): print(f"⚠️ บางโมเดลใช้ต้นไม้เกือบครบ {N_TREES} — ลองเพิ่ม LR หรือ N_TREES")

## 3. แปลงการกระจาย → ค่าที่ดีที่สุดสำหรับ MAE

In [ ]:
def pred_B(p, mu, thr=0.5):                       # hurdle + median ของคนจ่าย
    return np.where(p > thr, np.expm1(mu), 0.0)
def pred_C(p, mu, sig, scale=1.0, shift=0.0):     # ZILN quantile q = 1 − 1/(2p)
    q = np.clip(1 - 1 / (2 * np.clip(p + shift, 1e-6, 1)), 1e-4, 0.5)
    out = np.expm1(mu + scale * sig * norm.ppf(q))
    return np.where(p + shift > 0.5, np.clip(out, 0, 300000), 0.0)
def pred_D(p, Q, shift=0.0):                       # multi-quantile, interpolate ที่ q
    q = np.clip(1 - 1 / (2 * np.clip(p + shift, 1e-6, 1)), ALPHAS[0], ALPHAS[-1])
    v = np.array([np.interp(qi, ALPHAS, Qi) for qi, Qi in zip(q, Q)])
    return np.where(p + shift > 0.5, np.clip(np.expm1(v), 0, 300000), 0.0)

R = {"zero": nmae(y, np.zeros(n)), "A_mae": nmae(y, np.clip(O["mae"], 0, None)), "B_median": nmae(y, pred_B(O["p"], O["mu"]))}
R["C_ziln_theory"] = nmae(y, pred_C(O["p"], O["mu"], O["sig"]))
R["D_mquant_theory"] = nmae(y, pred_D(O["p"], O["q"]))
# ปรับ 1–2 พารามิเตอร์บน OOF (เล็กน้อย → optimistic นิดเดียว)
gC = max(((sc, sh) for sc in np.arange(0.6, 4.01, 0.2) for sh in np.arange(-0.1, 0.101, 0.025)),
         key=lambda t: nmae(y, pred_C(O["p"], O["mu"], O["sig"], *t)))
R["C_ziln_tuned"] = nmae(y, pred_C(O["p"], O["mu"], O["sig"], *gC))
gD = max(np.arange(-0.1, 0.101, 0.025), key=lambda sh: nmae(y, pred_D(O["p"], O["q"], sh)))
R["D_mquant_tuned"] = nmae(y, pred_D(O["p"], O["q"], gD))
print("best C (sigma scale, p shift):", np.round(gC, 3), "| best D shift:", round(float(gD), 3))
for k, v in R.items(): print(f"{k:18s} {v:.4f}")

## 4. Blend (A, C, D) บน OOF

In [ ]:
PA = np.clip(O["mae"], 0, None); PC = pred_C(O["p"], O["mu"], O["sig"], *gC); PD = pred_D(O["p"], O["q"], gD)
grid = [(wa, wc, round(1 - wa - wc, 2)) for wa in np.arange(0, 1.01, 0.1) for wc in np.arange(0, 1.01 - wa, 0.1)]
wbest = max(grid, key=lambda w: nmae(y, w[0]*PA + w[1]*PC + w[2]*PD))
R["E_blend"] = nmae(y, wbest[0]*PA + wbest[1]*PC + wbest[2]*PD)
print("blend weights (A, C, D):", np.round(wbest, 2), "| OOF NMAE", round(R["E_blend"], 4))
# วิเคราะห์ราย segment
seg = tr["segment"].fillna(-1).values; best_oof = wbest[0]*PA + wbest[1]*PC + wbest[2]*PD
print(pd.DataFrame({"seg": seg, "err": np.abs(y - best_oof), "y": y}).groupby("seg").agg(n=("y", "size"), sum_err=("err", "sum"), sum_y=("y", "sum")).assign(share_err=lambda d: (d.sum_err / d.sum_err.sum()).round(3)))

## 5. Test predictions + บันทึก

In [ ]:
TA = np.clip(T["mae"], 0, None); TC = pred_C(T["p"], T["mu"], T["sig"], *gC); TD = pred_D(T["p"], T["q"], gD)
choices = {"A_mae": TA, "C_ziln_tuned": TC, "D_mquant_tuned": TD, "E_blend": wbest[0]*TA + wbest[1]*TC + wbest[2]*TD}
best_name = max(choices, key=lambda k: R[k]); pred = np.round(choices[best_name], 2)
os.makedirs("/kaggle/working/preds", exist_ok=True)
pd.DataFrame({"id": ids, "pred": pred}).to_csv("/kaggle/working/preds/t3_ziln.csv", index=False)
for k, v in choices.items(): pd.DataFrame({"id": ids, "pred": np.round(v, 2)}).to_csv(f"/kaggle/working/preds/t3_{k}.csv", index=False)
pd.DataFrame({"y": y, "A": PA, "C": PC, "D": PD, "p": O["p"], "mu": O["mu"], "sig": O["sig"]}).to_csv("/kaggle/working/preds/t3_oof.csv", index=False)
json.dump({"oof_nmae": {k: round(float(v), 5) for k, v in R.items()}, "best": best_name, "blend_w": [float(w) for w in wbest],
           "C_params": [float(x) for x in gC], "D_shift": float(gD), "best_iter": best_it},
          open("/kaggle/working/preds/t3_ziln_summary.json", "w"), indent=1)
print("ใช้:", best_name, "| OOF NMAE", round(R[best_name], 4), "| test pred: zero%", round((pred == 0).mean(), 3), "mean", round(pred.mean(), 1))
print("เขียน /kaggle/working/preds/ แล้ว — ไม่ได้ submit อะไร")